In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import nibabel as nib
import pandas as pd

import warnings


HERE = os.getcwd()




### Samall trick becuase there is not cortical parcelation or label 383
Missing labels: [3, 383]

In [ ]:
INPUT_RECON_DIR = os.path.join(HERE, "data/inputs/recon_0.2mm")
CHANGED_INPUT_RECON_DIR = os.path.join(HERE, "data/inputs/changed_recon_0.2mm")
case_list = os.listdir(INPUT_RECON_DIR)

os.makedirs(CHANGED_INPUT_RECON_DIR, exist_ok=True)

for case in case_list:
    if case.endswith(".nii.gz"):
        print(case)
        case_path = os.path.join(INPUT_RECON_DIR, case)
        img = nib.load(case_path)
        img_data = img.get_fdata()
        img_affine = img.affine

        # convert label 3 to 2002 [first label of the cortex]
        img_data[img_data == 3] = 2002
        # convert label 383 to 50 [lateral ventricle]
        img_data[img_data == 383] = 50

        # convert to int16 to save space
        img_data = img_data.astype(np.int16)

        # save the modified with the same affine, header, and filename and data type
        new_img = nib.Nifti1Image(img_data, img_affine, img.header)
        new_case_path = os.path.join(CHANGED_INPUT_RECON_DIR, case)
        nib.save(new_img, new_case_path)


case-3.nii.gz
case-1.nii.gz
case-5.nii.gz
case-2.nii.gz
case-4.nii.gz


### Run mergin code

In [ ]:
"""Merge the recon_0.2mm segmentations at levels L1-L4 into data/outputs/level<n>.

Background -> 0, the other parents -> 1..N (alphabetical), labels missing from
the ontology (e.g. 383) -> N+1. Label 3 (whole cerebral cortex) is added to the
ontology as an extra row so it is merged with the cortex instead of being unknown.
"""

from merge_labels import load_ontology, merge_folder, valid_names

ONTOLOGY_CSV = os.path.join(HERE, "data/inputs/nextbrain_ontology_v1.csv")


INPUT_DIR = CHANGED_INPUT_RECON_DIR
OUTPUT_DIR = os.path.join(HERE, "data/outputs/changed_recon_0.2mm")

# INPUT_DIR = os.path.join(HERE, "data/inputs/MRIxFields_T1W_7T_001")
# OUTPUT_DIR = os.path.join(HERE, "data/outputs/MRIxFields_T1W_7T_001")



ontology = load_ontology(ONTOLOGY_CSV)
for n in (1, 2, 3, 4):
    level = f"L{n}"
    names = [name for name in valid_names(ontology, level) if name != "Background"]
    mapping = {"Background": 0, **{name: i for i, name in enumerate(names, 1)}}
    unknown_value = len(names) + 1
    print(f"\n=== {level}: {len(names)} parents, unknown -> {unknown_value}")
    merge_folder(INPUT_DIR, os.path.join(OUTPUT_DIR, f"level{n}"), ONTOLOGY_CSV, level, mapping,
                    unmapped_value=0, unknown_value=unknown_value)



=== L1: 8 parents, unknown -> 9
[ok] seg.left.nii.gz: labels [0, 1, 2, 3, 4, 5, 6, 7, 8]

=== L2: 18 parents, unknown -> 19
[ok] seg.left.nii.gz: labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

=== L3: 29 parents, unknown -> 30
[ok] seg.left.nii.gz: labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

=== L4: 72 parents, unknown -> 73
[ok] seg.left.nii.gz: labels [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 46, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72]


### Run code with extra things

In [ ]:
# """Merge the recon_0.2mm segmentations at levels L1-L4 into data/outputs/level<n>.

# Background -> 0, the other parents -> 1..N (alphabetical), labels missing from
# the ontology (e.g. 383) -> N+1. Label 3 (whole cerebral cortex) is added to the
# ontology as an extra row so it is merged with the cortex instead of being unknown.
# """
# import os
# import warnings

# from merge_labels import load_ontology, merge_folder, valid_names

# HERE = os.path.dirname(os.path.abspath(__file__))
# INPUT_DIR = os.path.join(HERE, "data/inputs/recon_0.2mm")
# OUTPUT_DIR = os.path.join(HERE, "data/outputs")
# ONTOLOGY_CSV = os.path.join(HERE, "data/inputs/nextbrain_ontology_v1.csv")

# EXTRA_LABELS = [{
#     "allen_id": 3,
#     "leaf_name": "cerebral_cortex",
#     "L1_tissue": "Cerebral GM",
#     "L2_division": "Cerebral cortex",
#     "L3_structure": "Cerebral-Cortex",
#     "L4_subregion": "Cerebral cortex (unassigned)",
# }]

# if __name__ == "__main__":
#     warnings.simplefilter("ignore", UserWarning)
#     ontology = load_ontology(ONTOLOGY_CSV, EXTRA_LABELS)
#     for n in (1, 2, 3, 4):
#         level = f"L{n}"
#         names = [name for name in valid_names(ontology, level) if name != "Background"]
#         mapping = {"Background": 0, **{name: i for i, name in enumerate(names, 1)}}
#         unknown_value = len(names) + 1
#         print(f"\n=== {level}: {len(names)} parents, unknown -> {unknown_value}")
#         merge_folder(INPUT_DIR, os.path.join(OUTPUT_DIR, f"level{n}"), ONTOLOGY_CSV, level, mapping,
#                      unmapped_value=0, unknown_value=unknown_value, extra_labels=EXTRA_LABELS)
